# Calibrated invariance criteria: RMSEA_D and dynamic (DMI) cutoffs, all scales × all pairs

Companion to `NB_conventional_criteria.ipynb`. That notebook judged every Wu–Estabrook
ladder step with the conventional fixed cutoffs (ΔCFI ≥ −.010, ΔRMSEA ≤ .015). This
notebook re-judges the same 20 scales × 3 pairs (HiTOP on val_gp / val_en / gp_en;
PHQ-8 / GAD-7 / BAARS on gp_en) with two calibrated alternatives and compares all of them
against the permutation-test verdicts.

**RMSEA_D** (Savalei, Brace & Fouladi, 2023, *Psychological Methods*, doi:10.1037/met0000537):
the RMSEA of the chi-square *difference* test rather than the difference of two RMSEAs.
Multiple-group form (Dudgeon, 2004; Beribisky & Hancock, 2024, doi:10.1177/00131644231202949):
`RMSEA_D = sqrt( G (ΔT − Δdf) / (Δdf (N − G)) )`, N = total sample size, G = 2.
With WLSMV the difference statistic is the *scaled* Δχ² from `lavaan::lavTestLRT`
(Satorra–Bentler-type scaling). lavaan's own `RMSEA` column in `lavTestLRT` uses the
unscaled Δχ²; it is recorded alongside for reference. No cutoff for RMSEA_D is established
in the literature; it is reported continuously and screened at the conventional RMSEA
benchmarks .05 and .08.

**Dynamic measurement-invariance cutoffs** (McNeish, 2026, *Psychological Methods*,
doi:10.1037/met0000767): for each ladder step, simulate data from the *constrained*
(invariant) model's estimates with the observed group sizes, fit the lower and higher
models to each replicate, and take the resulting null distribution of ΔCFI / ΔRMSEA as the
cutoff (observed ΔCFI below the 5th percentile, or ΔRMSEA above the 95th, rejects).
McNeish defines the method for continuous ML indicators; here it is extended to the
ordinal WLSMV / theta-parameterization models by simulating latent responses from the
fitted loadings, residual variances, factor means/variances and thresholds, and
categorising at the fitted thresholds (R function `dmi_null`, defined below, parallelised
with `parallel::mclapply`). 500 replications per transition.

Outputs: `data/cfa_calibrated_criteria/calibrated_screen.csv` (one row per scale × pair ×
ladder transition), `dmi_null_draws.pkl` (the simulated null draws), `summary_vs_permutation.csv`.

In [1]:
import os
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_var, "1")
import platform, math, time, datetime
from pathlib import Path

import numpy as np
import pandas as pd

import hitop_cfa
from hitop_cfa.r_env import ro, set_seeds, silence_r
from hitop_cfa import WU_ESTABROOK_LEVELS, fit_measeq_level
from hitop_cfa.measeq import fit_is_converged
from hitop_cfa.fit import push_model_to_r
from hitop_cfa.scales import OTHER_SCALES, ORIG_ITEMS

In [2]:
dat_dir = Path('../data/')
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa_calibrated_criteria'
cfa_dir.mkdir(exist_ok=True)
helpfile_dir = cfa_dir / 'temp'; helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'
progress_log = cfa_dir / 'progress.log'

total_cpus = os.cpu_count()
cpus_to_use = total_cpus - 2 if ('arm' in platform.machine().lower() or 'aarch' in platform.machine().lower()) else total_cpus // 2 - 1
DMI_REPS = 500
BASE_SEED = 12345
LEVELS = [name for name, _ in WU_ESTABROOK_LEVELS]
GE = dict(WU_ESTABROOK_LEVELS)

def log(msg):
    stamp = datetime.datetime.now().isoformat(timespec='seconds')
    line = f'[{stamp}] {msg}'
    print(line, flush=True)
    with open(progress_log, 'a') as fh:
        fh.write(line + '\n')

set_seeds(BASE_SEED)
silence_r()
log(f'cpus={cpus_to_use} DMI_REPS={DMI_REPS}')

[2026-10-01T16:32:56] cpus=14 DMI_REPS=500


In [3]:
data_val = pd.read_csv(fin_dir / 'dat_val.csv')
data_gp = pd.read_csv(fin_dir / 'dat_gp_grid1st_full.csv')
data_en = pd.read_csv(fin_dir / 'dat_en_grid1st_full.csv')
# reference group first in every pair (same orders as the rest of the pipeline)
datasets = {'val_gp': pd.concat([data_val, data_gp]),
            'val_en': pd.concat([data_val, data_en]),
            'gp_en': pd.concat([data_gp, data_en])}
ALL_SCALE_ITEMS = list(OTHER_SCALES.items()) + list(ORIG_ITEMS.items())
print({k: len(v) for k, v in datasets.items()})

{'val_gp': 993, 'val_en': 801, 'gp_en': 802}


## DMI null simulation (R, via rpy2)

`dmi_null(fit_con, syntax_lo, syntax_hi, items, nrep, cores, seed)` takes the fitted
*constrained* model of a ladder step as the data-generating model: for each group it draws
factor scores from N(α, Ψ), residuals from N(0, Θ), forms latent responses
y* = ν + Λη + ε, and categorises each item at that group's fitted thresholds, with the
observed group sizes. The lower and higher measEq.syntax models of the step are refit to
each replicate with the same WLSMV / theta settings as the real analysis, and
ΔCFI = CFI_hi − CFI_lo and ΔRMSEA = RMSEA_hi − RMSEA_lo (scaled versions) are returned.
Replicates whose fits fail or do not converge (e.g. a simulated group with an empty
category) return NA and are dropped before the percentiles are taken.

In [4]:
R_CODE = '\ndmi_null <- function(fit_con, syntax_lo, syntax_hi, items, nrep, cores, seed) {\n  est  <- lavaan::lavInspect(fit_con, "est")\n  nobs <- lavaan::lavInspect(fit_con, "nobs")\n  glab <- lavaan::lavInspect(fit_con, "group.label")\n  G <- length(est); p <- length(items)\n  gen_group <- function(g, n) {\n    e <- est[[g]]\n    L   <- as.matrix(e$lambda)[items, , drop = FALSE]\n    Th  <- as.matrix(e$theta)[items, items, drop = FALSE]\n    Psi <- as.matrix(e$psi)\n    alpha <- as.numeric(e$alpha)\n    nu  <- as.numeric(as.matrix(e$nu)[items, 1])\n    tau <- as.matrix(e$tau)\n    tau_item <- sub("\\\\|t[0-9]+$", "", rownames(tau))\n    k <- ncol(L)\n    eta <- matrix(rnorm(n * k), n, k) %*% chol(Psi)\n    eta <- sweep(eta, 2, alpha, "+")\n    eps <- matrix(rnorm(n * p), n, p) %*% chol(Th)\n    ystar <- matrix(nu, n, p, byrow = TRUE) + eta %*% t(L) + eps\n    y <- matrix(NA_integer_, n, p); colnames(y) <- items\n    for (j in seq_len(p)) {\n      tj <- sort(tau[tau_item == items[j], 1])\n      y[, j] <- findInterval(ystar[, j], tj) + 1L\n    }\n    out <- as.data.frame(y)\n    out$whichdata <- glab[g]\n    out\n  }\n  one_rep <- function(r) {\n    set.seed(seed + r)\n    dat <- do.call(rbind, lapply(seq_len(G), function(g) gen_group(g, nobs[g])))\n    dat$whichdata <- factor(dat$whichdata, levels = glab)\n    fit_fun <- function(syn) {\n      f <- try(suppressWarnings(lavaan::cfa(syn, data = dat, group = "whichdata",\n                                            ordered = items, parameterization = "theta",\n                                            estimator = "WLSMV")), silent = TRUE)\n      if (inherits(f, "try-error")) return(c(NA_real_, NA_real_))\n      if (!isTRUE(lavaan::lavInspect(f, "converged"))) return(c(NA_real_, NA_real_))\n      fm <- try(suppressWarnings(lavaan::fitMeasures(f, c("cfi.scaled", "rmsea.scaled"))), silent = TRUE)\n      if (inherits(fm, "try-error")) return(c(NA_real_, NA_real_))\n      as.numeric(fm)\n    }\n    lo <- fit_fun(syntax_lo); hi <- fit_fun(syntax_hi)\n    c(hi[1] - lo[1], hi[2] - lo[2])\n  }\n  res <- parallel::mclapply(seq_len(nrep), one_rep, mc.cores = cores, mc.set.seed = FALSE)\n  m <- do.call(rbind, res)\n  colnames(m) <- c("dcfi", "drmsea")\n  m\n}\n'
ro.r(R_CODE)
print("dmi_null defined in R")

dmi_null defined in R


In [5]:
def rmsea_d_mg(dT, ddf, N, G=2):
    """Multiple-group RMSEA of the chi-square difference test (Dudgeon 2004 form)."""
    if ddf <= 0 or not np.isfinite(dT):
        return float('nan')
    return math.sqrt(max(G * (dT - ddf) / (ddf * (N - G)), 0.0))

out_csv = cfa_dir / 'calibrated_screen.csv'
rows = []
nulls = {}
done_scales = set()
# resumable: a scale x pair cell counts as complete only when its strict row exists in the
# CSV and its strict null draws are in the pickle; partial cells are redone from scratch
if out_csv.exists():
    prev_df = pd.read_csv(out_csv)
    pkl = cfa_dir / 'dmi_null_draws.pkl'
    nulls = pd.read_pickle(pkl) if pkl.exists() else {}
    for (p_, s_), g_ in prev_df.groupby(['pair', 'scale']):
        if 'strict' in set(g_.level) and (p_, s_, 'strict') in nulls:
            done_scales.add((p_, s_))
    keep = [(p_, s_) in done_scales for p_, s_ in zip(prev_df.pair, prev_df.scale)]
    rows = prev_df[keep].to_dict('records')
    nulls = {k: v for k, v in nulls.items() if (k[0], k[1]) in done_scales}
    log(f'resume: {len(done_scales)} scale x pair cells already complete ({len(rows)} transitions)')
cell_ix = 0
t_start = time.time()
for pair, data in datasets.items():
    scale_items = ALL_SCALE_ITEMS if pair == 'gp_en' else list(ORIG_ITEMS.items())
    N = len(data); G = 2
    for scale, formula in scale_items:
        cell_ix += 1
        if (pair, scale) in done_scales:
            continue
        items = formula.split('=~', 1)[1].strip().split(' + ')
        push_model_to_r(scale, items, data, path_to_helpfile)
        fits_ok = {}
        for level in LEVELS:
            fit_measeq_level(items, GE[level], r_fit_name=f'f_{level}')
            fits_ok[level] = fit_is_converged(f'f_{level}')
            if not fits_ok[level]:
                log(f'{pair} {scale}: {level} model did not converge; ladder stops here')
                break
        prev = None
        for li, level in enumerate(LEVELS):
            if not fits_ok.get(level):
                break
            if prev is None:
                prev = level
                continue
            t0 = time.time()
            # --- observed statistics for this transition ---
            lrt = ro.r(f'tt <- lavaan::lavTestLRT(f_{prev}, f_{level}); '
                       f'c(tt[2, "Chisq diff"], tt[2, "Df diff"], tt[2, "Pr(>Chisq)"], tt[2, "RMSEA"])')
            dT, ddf, p_lrt, rmsea_lav = [float(x) for x in lrt]
            fm = ro.r(f'c(lavaan::fitMeasures(f_{prev}, c("cfi.scaled", "rmsea.scaled")), '
                      f'lavaan::fitMeasures(f_{level}, c("cfi.scaled", "rmsea.scaled")))')
            cfi_lo, rm_lo, cfi_hi, rm_hi = [float(x) for x in fm]
            dcfi = cfi_hi - cfi_lo
            drmsea = rm_hi - rm_lo
            rmsea_d = rmsea_d_mg(dT, ddf, N, G)
            # --- DMI null distribution under the constrained model of this step ---
            ro.globalenv['dmi_syntax_lo'] = ro.r(f'f_{prev}_mod_str')
            ro.globalenv['dmi_syntax_hi'] = ro.r(f'f_{level}_mod_str')
            ro.globalenv['dmi_items'] = ro.StrVector(items)
            seed = BASE_SEED + cell_ix * 100000 + li * 10000
            ro.r(f'dmi_res <- dmi_null(f_{level}, dmi_syntax_lo, dmi_syntax_hi, dmi_items, '
                 f'{DMI_REPS}, {cpus_to_use}, {seed})')
            null = np.column_stack([np.array(ro.r('as.numeric(dmi_res[, 1])')),
                                    np.array(ro.r('as.numeric(dmi_res[, 2])'))])
            nulls[(pair, scale, level)] = null
            ok = ~np.isnan(null).any(axis=1)
            nd = null[ok]
            if len(nd) >= 20:
                cfi_p05, cfi_p01 = np.percentile(nd[:, 0], [5, 1])
                rm_p95, rm_p99 = np.percentile(nd[:, 1], [95, 99])
            else:
                cfi_p05 = cfi_p01 = rm_p95 = rm_p99 = float('nan')
            row = dict(
                pair=pair, scale=scale, level=level, transition=f'{prev}->{level}',
                n_items=len(items), N=N,
                cfi_lo=round(cfi_lo, 4), cfi_hi=round(cfi_hi, 4),
                rmsea_lo=round(rm_lo, 4), rmsea_hi=round(rm_hi, 4),
                delta_cfi=round(dcfi, 4), delta_rmsea=round(drmsea, 4),
                chisq_diff_scaled=round(dT, 3), df_diff=int(ddf), p_lrt=p_lrt,
                rmsea_d=round(rmsea_d, 4), rmsea_d_lavaan_unscaled=round(rmsea_lav, 4),
                conv_pass_dCFI=bool(dcfi >= -0.010), conv_pass_dRMSEA=bool(drmsea <= 0.015),
                dmi_reps_ok=int(ok.sum()),
                dmi_cfi_p05=round(float(cfi_p05), 4), dmi_cfi_p01=round(float(cfi_p01), 4),
                dmi_rmsea_p95=round(float(rm_p95), 4), dmi_rmsea_p99=round(float(rm_p99), 4),
                dmi_pass_dCFI_05=bool(dcfi >= cfi_p05) if np.isfinite(cfi_p05) else None,
                dmi_pass_dRMSEA_05=bool(drmsea <= rm_p95) if np.isfinite(rm_p95) else None,
                dmi_pass_dCFI_01=bool(dcfi >= cfi_p01) if np.isfinite(cfi_p01) else None,
                dmi_pass_dRMSEA_01=bool(drmsea <= rm_p99) if np.isfinite(rm_p99) else None,
                seconds=round(time.time() - t0, 1),
            )
            rows.append(row)
            pd.DataFrame(rows).to_csv(out_csv, index=False)
            log(f'{pair} {scale} {prev}->{level}: dCFI={dcfi:+.4f} (DMI p05 {cfi_p05:+.4f}) '
                f'dRMSEA={drmsea:+.4f} (DMI p95 {rm_p95:+.4f}) RMSEA_D={rmsea_d:.4f} '
                f'p_lrt={p_lrt:.4f} reps_ok={int(ok.sum())} [{row["seconds"]}s]')
            prev = level
        pd.to_pickle(nulls, cfa_dir / 'dmi_null_draws.pkl')
cal = pd.DataFrame(rows)
log(f'DONE {len(cal)} transitions in {(time.time() - t_start) / 60:.1f} min')
cal

[2026-10-01T16:32:56] resume: 30 scale x pair cells already complete (120 transitions)


[2026-10-01T16:33:16] gp_en baars_inattention_sum configural->thresholds: dCFI=-0.0009 (DMI p05 -0.0006) dRMSEA=-0.0051 (DMI p95 +0.0064) RMSEA_D=0.0416 p_lrt=0.0849 reps_ok=500 [18.6s]


[2026-10-01T16:33:34] gp_en baars_inattention_sum thresholds->metric: dCFI=+0.0026 (DMI p05 -0.0009) dRMSEA=-0.0109 (DMI p95 +0.0121) RMSEA_D=0.0039 p_lrt=0.4288 reps_ok=500 [18.4s]


[2026-10-01T17:59:07] gp_en baars_inattention_sum metric->scalar: dCFI=+0.0013 (DMI p05 -0.0009) dRMSEA=-0.0069 (DMI p95 +0.0112) RMSEA_D=0.0296 p_lrt=0.2136 reps_ok=500 [5132.5s]


[2026-10-01T19:35:51] gp_en baars_inattention_sum scalar->strict: dCFI=+0.0019 (DMI p05 -0.0011) dRMSEA=-0.0078 (DMI p95 +0.0147) RMSEA_D=0.0377 p_lrt=0.1180 reps_ok=500 [5804.3s]


[2026-10-01T20:36:09] gp_en baars_hyperactivity_sum configural->thresholds: dCFI=-0.0013 (DMI p05 -0.0016) dRMSEA=-0.0157 (DMI p95 +0.0159) RMSEA_D=0.0000 p_lrt=0.7094 reps_ok=489 [3615.2s]


... (65 outputs left out)

[2026-10-02T09:40:23] gp_en well_being metric->scalar: dCFI=+0.0003 (DMI p05 -0.0008) dRMSEA=-0.0053 (DMI p95 +0.0121) RMSEA_D=0.0782 p_lrt=0.0003 reps_ok=500 [19.7s]


[2026-10-02T09:40:42] gp_en well_being scalar->strict: dCFI=+0.0055 (DMI p05 -0.0012) dRMSEA=-0.0151 (DMI p95 +0.0126) RMSEA_D=0.0193 p_lrt=0.3210 reps_ok=500 [18.3s]


[2026-10-02T09:40:42] DONE 192 transitions in 1027.8 min


,pair,scale,level,transition,n_items,N,cfi_lo,cfi_hi,rmsea_lo,rmsea_hi,...,dmi_reps_ok,dmi_cfi_p05,dmi_cfi_p01,dmi_rmsea_p95,dmi_rmsea_p99,dmi_pass_dCFI_05,dmi_pass_dRMSEA_05,dmi_pass_dCFI_01,dmi_pass_dRMSEA_01,seconds
0,val_gp,anhedonic_depression,thresholds,configural->thresholds,10,993,0.9757,0.9739,0.1038,0.1006,...,500,-0.0004,-0.0006,0.0054,0.0109,False,True,False,True,20.7
1,val_gp,anhedonic_depression,metric,thresholds->metric,10,993,0.9739,0.9755,0.1006,0.0925,...,500,-0.0007,-0.0013,0.0114,0.0167,True,True,True,True,21.1
2,val_gp,anhedonic_depression,scalar,metric->scalar,10,993,0.9755,0.9747,0.0925,0.0895,...,500,-0.0008,-0.0014,0.0106,0.0154,True,True,True,True,20.5
3,val_gp,anhedonic_depression,strict,scalar->strict,10,993,0.9747,0.9793,0.0895,0.0772,...,500,-0.0008,-0.0017,0.0119,0.0185,True,True,True,True,18.6
4,val_gp,anxious_worry,thresholds,configural->thresholds,7,993,0.9970,0.9960,0.0577,0.0596,...,500,-0.0003,-0.0005,0.0108,0.0194,False,True,False,True,15.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,gp_en,social_anxiety,strict,scalar->strict,10,802,0.9872,0.9908,0.0765,0.0616,...,500,-0.0010,-0.0017,0.0132,0.0192,True,True,True,True,19.2
188,gp_en,well_being,thresholds,configural->thresholds,10,802,0.9721,0.9709,0.1185,0.1134,...,500,-0.0005,-0.0007,0.0068,0.0123,False,True,False,True,20.2
189,gp_en,well_being,metric,thresholds->metric,10,802,0.9709,0.9747,0.1134,0.1002,...,500,-0.0010,-0.0019,0.0150,0.0230,True,True,True,True,20.5
190,gp_en,well_being,scalar,metric->scalar,10,802,0.9747,0.9750,0.1002,0.0948,...,500,-0.0008,-0.0013,0.0121,0.0190,True,True,True,True,19.7


## Comparison against the permutation verdicts

Reference: the permutation ladders (`data/cfa/orig_cfa_res.csv` for HiTOP, 3 pairs;
`data/cfa_other_gp_en/orig_cfa_res.csv` for PHQ/GAD/BAARS on gp_en). Each method's verdict
for a scale × pair is its *first failing delta level* (thresholds, metric, scalar or
strict); the configural level has no delta and is invisible to every delta-based method,
so permutation configural failures are tabulated separately. Methods compared:

- `conv`: ΔCFI < −.010 or ΔRMSEA > .015 (conventional)
- `lrt`: scaled Δχ² p < .05 (the parametric difference test)
- `rmsea_d_05`, `rmsea_d_08`: RMSEA_D > .05 / > .08
- `dmi05`: observed ΔCFI below the 5th percentile or ΔRMSEA above the 95th of the DMI null
- `dmi05_cfi`, `dmi05_rmsea`: the two DMI components alone
- `dmi01`: the stricter 1st / 99th percentile version

In [6]:
cal = pd.read_csv(out_csv)
DELTA_LEVELS = ['thresholds', 'metric', 'scalar', 'strict']
ORDER = {l: i for i, l in enumerate(LEVELS)}

# ---- permutation reference ----
perm_h = pd.read_csv(dat_dir / 'cfa' / 'orig_cfa_res.csv')
perm_o = pd.read_csv(dat_dir / 'cfa_other_gp_en' / 'orig_cfa_res.csv')
perm = pd.concat([perm_h, perm_o], ignore_index=True)
perm['pair'] = perm['pair'].map({'V_GP': 'val_gp', 'V_EN': 'val_en', 'GP_EN': 'gp_en'})
PCOLS = dict(zip(LEVELS, ['pconfig', 'pthresholds', 'pmetric', 'pscalar', 'pstrict']))
for c in PCOLS.values():
    perm[c] = pd.to_numeric(perm[c], errors='coerce')

def perm_first_fail(r):
    if pd.isna(r['pconfig']):
        return 'configural'
    for lev in LEVELS:
        p = r[PCOLS[lev]]
        if pd.isna(p):
            return lev          # non-convergence mid-ladder
        if p < 0.05:
            return lev
    return 'pass'
perm['perm_fail'] = perm.apply(perm_first_fail, axis=1)
perm = perm[['pair', 'scale', 'perm_fail']]

# ---- method verdicts from the calibrated screen ----
flags = {
    'conv':        lambda d: (~d.conv_pass_dCFI) | (~d.conv_pass_dRMSEA),
    'lrt':         lambda d: d.p_lrt < 0.05,
    'rmsea_d_05':  lambda d: d.rmsea_d > 0.05,
    'rmsea_d_08':  lambda d: d.rmsea_d > 0.08,
    'dmi05':       lambda d: (d.dmi_pass_dCFI_05 == False) | (d.dmi_pass_dRMSEA_05 == False),
    'dmi05_cfi':   lambda d: d.dmi_pass_dCFI_05 == False,
    'dmi05_rmsea': lambda d: d.dmi_pass_dRMSEA_05 == False,
    'dmi01':       lambda d: (d.dmi_pass_dCFI_01 == False) | (d.dmi_pass_dRMSEA_01 == False),
}
verdicts = perm.copy()
for name, fn in flags.items():
    d = cal.copy(); d['flag'] = fn(d).fillna(False).astype(bool)
    first = (d[d.flag].sort_values('level', key=lambda s: s.map(ORDER))
               .groupby(['pair', 'scale'])['level'].first())
    verdicts[name] = verdicts.set_index(['pair', 'scale']).index.map(first.to_dict()).fillna('pass') if False else \
        [first.get((p, s), 'pass') for p, s in zip(verdicts.pair, verdicts.scale)]

verdicts['perm_delta_fail'] = verdicts.perm_fail.isin(DELTA_LEVELS)
verdicts['perm_config_fail'] = verdicts.perm_fail == 'configural'
verdicts['perm_pass'] = verdicts.perm_fail == 'pass'
print(f"cells: {len(verdicts)}; permutation: {int(verdicts.perm_delta_fail.sum())} fail at a delta level, "
      f"{int(verdicts.perm_config_fail.sum())} fail configural, {int(verdicts.perm_pass.sum())} pass strict")

summary = []
for name in flags:
    v = verdicts
    detected = v.perm_delta_fail & (v[name] != 'pass')
    exact = detected & (v[name] == v.perm_fail)
    config_flagged_later = v.perm_config_fail & (v[name] != 'pass')
    false_flag = v.perm_pass & (v[name] != 'pass')
    summary.append(dict(
        method=name,
        detected_of_delta_failures=f"{int(detected.sum())}/{int(v.perm_delta_fail.sum())}",
        exact_level_agreement=int(exact.sum()),
        configural_failures_flagged_at_some_level=f"{int(config_flagged_later.sum())}/{int(v.perm_config_fail.sum())}",
        flags_on_permutation_passing_cells=f"{int(false_flag.sum())}/{int(v.perm_pass.sum())}",
    ))
    for pair in ['val_gp', 'val_en', 'gp_en']:
        vp = v[v.pair == pair]
        det = vp.perm_delta_fail & (vp[name] != 'pass')
        summary[-1][f'{pair}_detected'] = f"{int(det.sum())}/{int(vp.perm_delta_fail.sum())}"
summary = pd.DataFrame(summary)
summary.to_csv(cfa_dir / 'summary_vs_permutation.csv', index=False)
verdicts.to_csv(cfa_dir / 'verdicts_by_method.csv', index=False)
pd.set_option('display.width', 250)
print(summary.to_string(index=False))
print()
print(verdicts.sort_values(['pair', 'scale']).to_string(index=False))

cells: 48; permutation: 29 fail at a delta level, 11 fail configural, 8 pass strict
     method detected_of_delta_failures  exact_level_agreement configural_failures_flagged_at_some_level flags_on_permutation_passing_cells val_gp_detected val_en_detected gp_en_detected
       conv                      19/29                     12                                      2/11                                1/8             7/8            8/12            4/9
        lrt                      29/29                     29                                     10/11                                4/8             8/8           12/12            9/9
 rmsea_d_05                      29/29                     24                                     10/11                                4/8             8/8           12/12            9/9
 rmsea_d_08                      26/29                     15                                      7/11                                0/8             7/8           12/12  

## Notes

- `rmsea_d` uses the scaled Δχ² (the statistic the permutation and parametric tests also
  use); `rmsea_d_lavaan_unscaled` is lavaan's `lavTestLRT` RMSEA column, which applies the
  same multigroup formula to the unscaled difference. They diverge wherever the scaling
  factor is far from 1.
- DMI percentiles come from replicates that converged under both models; `dmi_reps_ok`
  records how many of the 500 did. Transitions with fewer than 20 usable replicates get NaN
  cutoffs.
- The DMI extension to ordinal indicators simulates from the fitted constrained model
  exactly as estimated (including group-2 factor means/variances and residual variances
  where those are free at that level); McNeish's continuous-ML procedure fixes the group-2
  latent mean and variance. This is the natural analogue under Wu–Estabrook
  identification but is not the published procedure.